# V3 P2 extensions

Notebook này chạy riêng expanding quarterly walk-forward và audit transaction cost. Không ghi đè `v3_final_handoff`.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, os
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/kltn')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/maiphuowng205/kltn.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'scikit-learn', 'cvxpy'], check=True)
print('Repo:', REPO)

In [ ]:
# Nếu workspace từ notebook 00-07 còn tồn tại, dùng lại để tránh copy lại dataset.
WORKSPACE = Path('/content/vn_v3_workspace')
DATA_ROOT = WORKSPACE / 'data' / 'lseg_v3'
DRIVE_DATA = Path('/content/drive/MyDrive/kltn/frozen/vn_v3_lseg_2026-08-03/data/lseg_v3')
if not (DATA_ROOT / 'curated' / 'daily_panel.parquet').exists():
    if not DRIVE_DATA.exists():
        raise FileNotFoundError('Không tìm thấy dataset. Kiểm tra DRIVE_DATA và sửa đúng đường dẫn Google Drive.')
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATA, DATA_ROOT, dirs_exist_ok=True)
required = [DATA_ROOT / 'curated' / 'daily_panel.parquet', DATA_ROOT / 'curated' / 'universe_weekly.parquet', DATA_ROOT / 'model_ready' / 'weekly_features_targets.parquet']
for path in required:
    print(path, 'exists=', path.exists(), 'bytes=', path.stat().st_size if path.exists() else None)
    if not path.exists(): raise FileNotFoundError(path)
print('Dataset ready:', DATA_ROOT)

In [ ]:
# 1) Expanding quarterly Ridge walk-forward + cutoff audit
WALK_RUN = WORKSPACE / 'runs' / 'v3_extension_p2' / 'quarterly_walk_forward'
subprocess.run([sys.executable, str(REPO / 'scripts' / 'run_v3_walk_forward.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(WALK_RUN)], check=True)
import json, pandas as pd
print(json.loads((WALK_RUN / 'cutoff_audit.json').read_text()))
display(pd.read_parquet(WALK_RUN / 'retraining_checkpoints.parquet'))

In [ ]:
# 2) Transaction-cost availability audit
COST_RUN = WORKSPACE / 'runs' / 'v3_extension_p2' / 'cost_validation'
subprocess.run([sys.executable, str(REPO / 'scripts' / 'audit_v3_transaction_cost_rule.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(COST_RUN)], check=True)
print((COST_RUN / 'transaction_cost_validation.json').read_text())

In [ ]:
# 3) Tạo handoff P2 riêng
P2_HANDOFF = WORKSPACE / 'runs' / 'v3_p2_handoff'
subprocess.run([sys.executable, str(REPO / 'scripts' / 'assemble_v3_p2_handoff.py'), '--extension-root', str(WORKSPACE / 'runs' / 'v3_extension_p2'), '--output', str(P2_HANDOFF)], check=True)
print((P2_HANDOFF / 'handoff_manifest.json').read_text())

In [ ]:
# 4) Lưu handoff vào Google Drive để không mất khi Colab ngắt runtime
DRIVE_OUTPUT = Path('/content/drive/MyDrive/kltn/outputs/v3_p2_handoff')
shutil.copytree(P2_HANDOFF, DRIVE_OUTPUT, dirs_exist_ok=True)
print('Đã lưu P2 handoff tại:', DRIVE_OUTPUT)
print('Files:', len(list(DRIVE_OUTPUT.rglob('*'))))

## Diễn giải

Walk-forward phải có `status=PASS` và `total_future_rows_used=0`. Cost audit có thể báo `BLOCKED_NO_OBSERVED_QUOTES`; điều đó là kết quả hợp lệ vì frozen V3 chỉ có cost imputed, không có bid/ask hoặc tick data quan sát.